# House 1 Raw Data Cleaning — Full Pipeline Walkthrough
## 01 · Raw Data Cleaning Notebook

**Dataset:** REFIT Smart Home Dataset — House 1 (active power, Watts)  
**Licence:** Creative Commons Attribution 4.0 International (CC-BY 4.0)  
**Citation:** Murray, D., Stankovic, L., Stankovic, V. (2017). *An electrical load measurements dataset of United Kingdom households from a two-year longitudinal study.* Scientific Data, 4, 160122. https://doi.org/10.1038/sdata.2016.122

### Purpose
This notebook documents the complete data-cleaning pipeline for House 1 of the REFIT dataset.  
The pipeline converts two overlapping raw CSVs (~480 MB total, ~8.5 M rows combined) into a single  
1-minute-resolution cleaned CSV (`CLEAN_House1_1min.csv`, ~192 MB, 920 031 minute-bins).

> **Note:** The heavy pipeline has already been executed; this notebook loads pre-computed audit JSON  
> (`outputs/metrics/house1_audit.json`) and the cleaned CSV for validation and illustration.  
> Small-sample demonstrations use `src/cleaning.py` functions interactively.

---


In [ ]:
# --- Imports & path configuration ---
import sys
import json
import inspect
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display, Image, HTML

# Project root: one level above notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
print(f"PROJECT_ROOT : {PROJECT_ROOT}")

# Ensure src/ is on PYTHONPATH
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# --- Key paths ---
RAW_DIR     = PROJECT_ROOT / "data" / "raw"
OUT_DIR     = PROJECT_ROOT / "outputs"
RESULTS_DIR = PROJECT_ROOT / "results"
AUDIT_JSON  = OUT_DIR / "metrics" / "house1_audit.json"
CLEAN_CSV   = OUT_DIR / "cleaned" / "CLEAN_House1_1min.csv"

RAW_P1 = RAW_DIR / "RAW_House1_Part1.csv"
RAW_P2 = RAW_DIR / "RAW_House1_Part2.csv"

# --- Channel mapping (from REFIT README + cleaning.py) ---
CHANNEL_MAP = {
    "Aggregate"  : "Whole-home aggregate",
    "Appliance1" : "Fridge (Hotpoint RLA50P)",
    "Appliance2" : "Freezer-1 (Beko CF393APW)",
    "Appliance3" : "Freezer-2",
    "Appliance4" : "Washer Dryer (Creda T522VW)",
    "Appliance5" : "Washing Machine (Beko WMC6140)",
    "Appliance6" : "Dishwasher (Bosch)",
    "Appliance7" : "Computer (Lenovo)",
    "Appliance8" : "Television Site",
    "Appliance9" : "Electric Heater",
}

# --- Verify raw files exist and print sizes ---
print("\n=== Raw file inventory ===")
for p in [RAW_P1, RAW_P2]:
    size_mb = p.stat().st_size / 1e6 if p.exists() else -1
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {p.name}  ({size_mb:.1f} MB)")

print(f"\n  Audit JSON : {'✓' if AUDIT_JSON.exists() else '✗ MISSING'}  {AUDIT_JSON.name}")
print(f"  Cleaned CSV: {'✓' if CLEAN_CSV.exists() else '✗ MISSING'}  {CLEAN_CSV.name}  ({CLEAN_CSV.stat().st_size/1e6:.1f} MB if exists)")


---
## Section 2 — Data Discovery & File Inspection

The REFIT House 1 dataset ships as two overlapping CSV files:

| File | Rows | Size | UTC span |
|------|-----:|-----:|---------|
| `RAW_House1_Part1.csv` | 4,377,877 | 243.6 MB | 2013-10-09 → 2014-10-01 |
| `RAW_House1_Part2.csv` | 4,140,886 | 239.6 MB | 2014-06-27 → 2015-07-10 |

Both files share the same schema: `Time, Unix, Aggregate, Appliance1 … Appliance9`.

**Channel mapping for House 1:**

| Channel | Appliance |
|---------|-----------|
| Aggregate | Whole-home aggregate |
| Appliance1 | Fridge (Hotpoint RLA50P) |
| Appliance2 | Freezer-1 (Beko CF393APW) |
| Appliance3 | Freezer-2 |
| Appliance4 | Washer Dryer (Creda T522VW) |
| Appliance5 | Washing Machine (Beko WMC6140) |
| Appliance6 | Dishwasher (Bosch) |
| Appliance7 | Computer (Lenovo) |
| Appliance8 | Television Site |
| Appliance9 | Electric Heater |


In [ ]:
# --- Peek at the first 5 rows of each raw file (header only) ---
for label, path in [("Part 1", RAW_P1), ("Part 2", RAW_P2)]:
    print(f"\n{'='*60}")
    print(f"  {label}: {path.name}  ({path.stat().st_size:,} bytes)")
    print(f"{'='*60}")
    sample = pd.read_csv(path, nrows=5)
    print("  Columns:", list(sample.columns))
    print("  Dtypes:\n", sample.dtypes.to_string())
    print("  First 5 rows:")
    display(sample)

# Channel map as a formatted table
print("\n=== Channel Mapping ===")
display(pd.DataFrame(list(CHANNEL_MAP.items()), columns=["Channel", "Appliance"]).set_index("Channel"))


---
## Section 3 — Timestamp Parsing & Ordering

### Pre-computed results (from audit JSON)

| Metric | Part 1 | Part 2 |
|--------|-------:|-------:|
| Invalid Unix timestamps | 0 | 0 |
| Invalid Time strings | 0 | 0 |
| `time_unix_nonnull_mismatches` | 2,134,001 | 2,018,842 |
| `row_order_unix_decreases` | 1 | 0 |
| `row_order_unix_repeats` | 1,003,950 | 14,755 |

#### BST offset artefact
The `time_unix_nonnull_mismatches` are rows where the human-readable `Time` string  
disagrees with the `Unix` column by exactly **±1 second** — a known artefact of  
British Summer Time (BST = UTC+1).  During BST, the REFIT logger appears to have stored  
the local-time string without converting to UTC, while the Unix epoch is correct.  
**Resolution:** always use `Unix` as the authoritative timestamp.


In [ ]:
# --- Load audit JSON ---
with open(AUDIT_JSON) as f:
    audit = json.load(f)

p1_ts = audit["files"]["RAW_House1_Part1.csv"]
p2_ts = audit["files"]["RAW_House1_Part2.csv"]

ts_summary = pd.DataFrame({
    "Part 1": {
        "Invalid Unix":               p1_ts["invalid_unix_timestamps"],
        "Invalid Time strings":        p1_ts["invalid_time_strings"],
        "Time≠Unix mismatches (BST)":  p1_ts["time_unix_nonnull_mismatches"],
        "Row order decreases":         p1_ts["row_order_unix_decreases"],
        "Row order repeats (=0)":      p1_ts["row_order_unix_repeats"],
    },
    "Part 2": {
        "Invalid Unix":               p2_ts["invalid_unix_timestamps"],
        "Invalid Time strings":        p2_ts["invalid_time_strings"],
        "Time≠Unix mismatches (BST)":  p2_ts["time_unix_nonnull_mismatches"],
        "Row order decreases":         p2_ts["row_order_unix_decreases"],
        "Row order repeats (=0)":      p2_ts["row_order_unix_repeats"],
    }
})
display(ts_summary)

# --- Live demonstration on 500 000-row sample from Part 1 ---
print("\nLoading 500 000-row sample from Part 1 for live demonstration …")
from cleaning import inspect_timestamps

sample_p1 = pd.read_csv(RAW_P1, nrows=500_000)
ts_diag = inspect_timestamps(sample_p1)
print("\nLive timestamp diagnostics (500 k-row sample, Part 1):")
for k, v in ts_diag.items():
    print(f"  {k:35s}: {v}")


---
## Section 4 — Duplicate Timestamps

REFIT House 1 Part 1 has a severe duplicate problem: **1,833,597 rows** (42 %) share  
their Unix timestamp with at least one other row, forming **829,193 groups**.  
Of those groups, **118,191** are *conflicting* — the grouped rows disagree on at least  
one power channel value.

| Metric | Part 1 | Part 2 |
|--------|-------:|-------:|
| Exact duplicate rows | 595,935 | 0 |
| Rows at duplicate timestamps | 1,833,597 | 29,510 |
| Duplicate timestamp groups | 829,193 | 14,755 |
| Conflicting groups (any channel) | 118,191 | 14,362 |

### Why do conflicts occur?
Part 1 was written in two overlapping database passes; Part 2 is the later "corrected" pass.  
Conflicts arise when the two passes stored different sensor readings for the same second  
(sensor retransmissions, DB merge artefacts).

**Resolution policy:** stable-sort by Unix then original row order; for each channel at a  
duplicate timestamp keep the **last non-missing value**.  


In [ ]:
# --- Conflicting groups per channel from audit JSON ---
p1_conflict = p1_ts["conflicting_duplicate_timestamp_groups_by_channel"]
p2_conflict = p2_ts["conflicting_duplicate_timestamp_groups_by_channel"]

conflict_df = pd.DataFrame({"Part 1 conflicts": p1_conflict, "Part 2 conflicts": p2_conflict})
conflict_df.index.name = "Channel"
print("Conflicting duplicate timestamp groups by channel:")
display(conflict_df)

# --- Show duplicate window plot ---
dup_fig = PROJECT_ROOT / "outputs" / "figures" / "house1_duplicate_timestamp_window.png"
print(f"\nDuplicate timestamp window plot: {dup_fig}")
display(Image(filename=str(dup_fig)))


---
## Section 5 — Sampling Intervals, Missingness & Outages

The documented cadence is **8 seconds**, but the real inter-sample interval is variable.  
Median observed interval: **7 s** (both files).

### Gaps > 60 seconds

| File | Unique timestamps | Gaps > 60 s | Largest gap |
|------|------------------:|------------:|------------:|
| Part 1 | 3,373,473 | 781 | 3,594,414 s (41.6 days) |
| Part 2 | 4,126,131 | 3,604 | 747,251 s (8.6 days) |

### Outages in 1-min output
- **Total minute bins:** 920,031  
- **Missing aggregate bins:** 119,680 (13.0 %)  
- **Longest outage:** 59,906 minutes (~41.6 days) from **2014-01-24T01:57Z** to **2014-03-06T16:22Z**  
  (the system was completely offline for most of February 2014)

> **Imputation policy: none.** Missing minute bins remain NaN; downstream models must handle them.


In [ ]:
# --- Interval stats from audit JSON ---
resampled = audit.get("resampled_1min", {})

# Show the sampling/missingness figure
samp_fig = PROJECT_ROOT / "outputs" / "figures" / "house1_sampling_and_missingness.png"
print(f"Sampling & missingness plot: {samp_fig}")
display(Image(filename=str(samp_fig)))

# Outage summary from audit
print("\n=== Outage / Missing-run summary (Aggregate channel) ===")
agg_runs = audit["resampled_1min"]["missing_runs"]["Aggregate"]
for k, v in agg_runs.items():
    print(f"  {k:30s}: {v}")

# Show longest-gap figure
gap_fig = PROJECT_ROOT / "outputs" / "figures" / "house1_longest_gap.png"
print(f"\nLongest gap plot: {gap_fig}")
display(Image(filename=str(gap_fig)))


---
## Section 6 — Channel-specific Quality

The table below is derived from the raw-file audit.  Values are **raw** (before cleaning).

Key observations:
- **No missing values** in raw files — all channels populated on every row (Part 1 & 2).
- **No negative readings** in either file.
- **Appliance9 (Electric Heater):** Only 874 zero-readings in Part 1 — dramatically fewer  
  zeros than other appliances — suggesting it may always draw some standby power.  
  It also has the worst post-cleaning missingness (36.8 % of minute bins missing) due to  
  late recording start and the February 2014 outage.
- **High-power (>10 kW) readings** exist for most channels; these are flagged but **not removed**  
  (they may be genuine; we apply a note-only screen at 10,000 W).


In [ ]:
channels = ["Aggregate"] + [f"Appliance{i}" for i in range(1, 10)]

def channel_quality_table(file_key: str) -> pd.DataFrame:
    fd = audit["files"][file_key]
    rows = []
    for ch in channels:
        rows.append({
            "Channel": ch,
            "Appliance": CHANNEL_MAP[ch],
            "Missing (raw)": fd["raw_missing_by_channel"].get(ch, "—"),
            "Zero (raw)":    fd["raw_zero_by_channel"].get(ch, "—"),
            "Negative":      fd["raw_negative_by_channel"].get(ch, "—"),
            ">10 kW":        fd["raw_over_10000_w_by_channel"].get(ch, "—"),
            "Min W":         fd.get("raw_min_by_channel", {}).get(ch, "—"),
            "Max W":         fd.get("raw_max_by_channel", {}).get(ch, "—"),
        })
    return pd.DataFrame(rows).set_index("Channel")

print("=== Channel Quality — Part 1 ===")
display(channel_quality_table("RAW_House1_Part1.csv"))

print("\n=== Channel Quality — Part 2 ===")
display(channel_quality_table("RAW_House1_Part2.csv"))

print("\n=== Post-cleaning missing minute-bins per channel ===")
missing_1min = audit["resampled_1min"]["per_channel_missing_bins"]
total_bins = audit["resampled_1min"]["total_minute_bins"]
missing_df = pd.DataFrame([
    {"Channel": ch, "Appliance": CHANNEL_MAP[ch],
     "Missing bins": missing_1min[ch],
     "Missing %": f"{100*missing_1min[ch]/total_bins:.1f}%"}
    for ch in channels
]).set_index("Channel")
display(missing_df)
print("\nNote: Appliance9 (Electric Heater) is missing 36.8% of bins — worst of all channels.")


---
## Section 7 — Part 1 / Part 2 Overlap Analysis

The two files overlap in time between  
`Unix 1403906269 – 1412126827`  ≡  **2014-06-27 to 2014-10-01** (UTC).

| Metric | Count |
|--------|------:|
| Part 1 rows in overlap window | 1,257,865 |
| Part 2 rows in overlap window | 945,403 |
| Part 1 grouped unique timestamps | 835,888 |
| Part 2 grouped unique timestamps | 942,783 |
| Timestamps in **both** files | 191,889 |
| Timestamps in **Part 1 only** | 643,999 |
| Timestamps in **Part 2 only** | 750,894 |
| Shared: aggregate **matching** | 36,988 |
| Shared: aggregate **conflicting** | 144,267 |

### Interpretation
Of the 191,889 timestamps that appear in both files, **75.2 % have conflicting aggregate readings**  
(144,267 / 191,889). This confirms Part 2 is a corrected re-issue of Part 1, not simply an extension.

**Cleaning strategy:** Use **Part 2** as authoritative throughout its coverage.  
Part 1 contributes only the rows **strictly before the UTC minute** containing Part 2's first timestamp  
(`1403906220` — the start of the minute at Unix 1403906269).


In [ ]:
# --- Overlap analysis from audit JSON ---
overlap = audit["overlap_analysis"]

overlap_summary = pd.DataFrame([
    {"Metric": k, "Value": v}
    for k, v in overlap.items()
    if isinstance(v, (int, float, str))
])
print("=== Overlap Analysis ===")
display(overlap_summary.set_index("Metric"))

# Visualise merge outcome as a bar chart
merge_counts = {
    "Both":       overlap.get("timestamp_merge_both", 191889),
    "Part1 only": overlap.get("timestamp_merge_left_only", 643999),
    "Part2 only": overlap.get("timestamp_merge_right_only", 750894),
}
fig, ax = plt.subplots(figsize=(7, 3))
bars = ax.bar(list(merge_counts.keys()), list(merge_counts.values()),
              color=["#4472C4", "#ED7D31", "#A9D18E"])
ax.set_title("Overlap-window timestamp merge outcome")
ax.set_ylabel("Unique timestamp groups")
for bar, val in zip(bars, merge_counts.values()):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000,
            f"{val:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


---
## Section 8 — Cleaning Pipeline

The pipeline in `src/cleaning.py` applies the following steps in order:

1. **Load Part 1** — read full raw CSV; tag `_source_row`.  
2. **Trim Part 1** — keep only rows with `Unix < 1403906220` (minute boundary of Part 2 start).  
3. **Remove exact duplicates (Part 1)** — drop exact full-row duplicates (321,692 removed).  
4. **Timestamp-resolution deduplication (Part 1)** — stable-sort by Unix; for each duplicate  
   timestamp group keep last non-missing value per channel (260,732 secondary rows removed).  
5. **Load Part 2** — read full raw CSV.  
6. **Remove exact duplicates (Part 2)** — 0 removed.  
7. **Timestamp-resolution deduplication (Part 2)** — 14,755 secondary rows removed.  
8. **Concatenate** — Part 1 cleaned + Part 2 cleaned, chronologically.  
9. **Resample to 1 minute** — mean of observed readings within each UTC minute; empty → NaN.  
10. **Validate** — run quality checks; write `house1_audit.json`.  

### Cleaning statistics

| Step | Part 1 | Part 2 |
|------|-------:|-------:|
| Rows considered | 3,120,004 | 4,140,886 |
| Exact duplicates removed | 321,692 | 0 |
| Timestamp-resolution removed | 260,732 | 14,755 |
| Unique timestamps after dedup | 2,537,580 | 4,126,131 |
| Minute bins produced | 376,311 | 543,720 |
| **Total minute bins** | **920,031** | |


In [ ]:
# --- Show cleaning.py function signatures ---
import cleaning

funcs = [
    "read_raw_file", "inspect_timestamps", "diagnose_duplicates",
    "resolve_duplicate_timestamps", "resample_to_1min", "validate_cleaned",
]
print("=== Key functions in src/cleaning.py ===\n")
for name in funcs:
    fn = getattr(cleaning, name, None)
    if fn:
        sig = inspect.signature(fn)
        doc_first = (inspect.getdoc(fn) or "").split("\n")[0]
        print(f"  {name}{sig}")
        print(f"    └─ {doc_first}\n")

# --- Cleaning stats table from audit JSON ---
cleaning_stats = {
    "rows_considered":           [p1_ts.get("rows_considered", 3120004),   p2_ts.get("rows_considered", 4140886)],
    "exact_duplicates_removed":  [p1_ts.get("exact_duplicate_rows", 595935), p2_ts.get("exact_duplicate_rows", 0)],
    "ts_resolution_removed":     [p1_ts.get("ts_resolution_removed", 260732), p2_ts.get("ts_resolution_removed", 14755)],
    "unique_ts_after_dedup":     [p1_ts.get("unique_ts_after_resolution", 2537580), p2_ts.get("unique_ts_after_resolution", 4126131)],
    "minute_bins":               [376311, 543720],
}
cs_df = pd.DataFrame(cleaning_stats, index=["Part 1", "Part 2"]).T
cs_df.index.name = "Step"
print("\n=== Cleaning Statistics ===")
display(cs_df)


---
## Section 9 — Resampling to 1-Minute Resolution

The cleaned per-second data is aggregated into 1-minute bins using **mean** resampling.  
Empty minute bins remain `NaN` (no imputation).

### 1-minute output summary

| Metric | Value |
|--------|-------|
| Total minute bins | 920,031 |
| UTC start | 2013-10-09T14:06:00Z |
| UTC end | 2015-07-10T11:56:00Z |
| Bins with Aggregate reading | 800,351 (87.0 %) |
| Missing aggregate bins | 119,680 (13.0 %) |
| Appliance9 missing bins | 338,427 (36.8 %) |

The resampled file (`CLEAN_House1_1min.csv`) uses a **UTC DatetimeIndex** and floating-point power  
columns (Watts). Values that map to empty minute bins are `NaN`.


In [ ]:
# --- Quick peek at cleaned CSV (1000 rows, fast) ---
print("Loading first 1,000 rows of cleaned CSV for display …")
clean_sample = pd.read_csv(CLEAN_CSV, nrows=1000, parse_dates=["Time"])
print(f"Shape of sample: {clean_sample.shape}")
print(f"Columns: {list(clean_sample.columns)}")
print("\nFirst 5 rows:")
display(clean_sample.head())
print("\nLast 5 rows of sample:")
display(clean_sample.tail())

print("\nData types:")
display(clean_sample.dtypes)

print(f"\nNull counts in sample (1000 rows):")
display(clean_sample.isnull().sum())

# --- Display the before/after aggregate figure ---
agg_fig = PROJECT_ROOT / "results" / "figures" / "house1_aggregate_before_after.png"
print(f"\nAggregate before/after: {agg_fig}")
display(Image(filename=str(agg_fig)))


---
## Section 10 — Gap Handling & Outage Flags

### Imputation policy: **None**
No imputation is applied. Missing minute bins remain `NaN` to avoid introducing  
synthetic power readings that could corrupt NILM algorithm training.

### Missing-run summary (all channels)

All channels share the same **longest outage: 59,906 minutes** (2014-01-24 → 2014-03-06),  
confirming the meter was offline rather than individual appliances being off.

| Channel | Missing bins | Missing % | Runs | Longest run (min) |
|---------|------------:|----------:|-----:|------------------:|
| Aggregate   | 119,680 | 13.0 % | 1,018 | 59,906 |
| Appliance1  | 123,730 | 13.4 % | 2,205 | 59,906 |
| Appliance2  | 123,473 | 13.4 % | 2,029 | 59,906 |
| Appliance3  | 121,143 | 13.2 % | 1,057 | 59,906 |
| Appliance4  | 132,757 | 14.4 % | 2,706 | 59,906 |
| Appliance5  | 129,681 | 14.1 % | 2,887 | 59,906 |
| Appliance6  | 120,560 | 13.1 % | 966 | 59,906 |
| Appliance7  | 119,674 | 13.0 % | 1,018 | 59,906 |
| Appliance8  | 121,879 | 13.2 % | 1,679 | 59,906 |
| Appliance9  | 338,427 | 36.8 % | 1,734 | 59,906 |


In [ ]:
# --- Missing-run table from audit JSON ---
missing_runs_data = audit["resampled_1min"]["missing_runs"]
total_bins = audit["resampled_1min"]["total_minute_bins"]

run_rows = []
for ch in channels:
    r = missing_runs_data[ch]
    run_rows.append({
        "Channel":         ch,
        "Appliance":       CHANNEL_MAP[ch],
        "Missing bins":    r["missing_bins"],
        "Missing %":       f"{100*r['missing_bins']/total_bins:.1f}%",
        "Runs":            r["missing_runs"],
        "Longest run (min)": r["longest_run_minutes"],
        "Longest start":   r["longest_run_start_utc"],
        "Longest end":     r["longest_run_end_utc"],
    })

run_df = pd.DataFrame(run_rows).set_index("Channel")
display(run_df)

# Show longest-gap figure
gap_fig = PROJECT_ROOT / "outputs" / "figures" / "house1_longest_gap.png"
display(Image(filename=str(gap_fig)))


---
## Section 11 — Validation & Quality Tables

### Sum-of-appliances check
On the 576,424 minute bins where **all channels are present**, the sum of the 9 appliances  
is compared to the aggregate channel:

| Metric | Value |
|--------|-------|
| Complete-channel minute bins | 576,424 |
| Median (appliance_sum − aggregate) | −159.4 W |
| Mean absolute difference | 354.3 W |
| Exactly equal (sum = aggregate) | 1 |

The median offset of −159 W indicates systematically **more power in the aggregate than  
the sum of sub-meters** — consistent with unmeasured loads (e.g. lights, sockets not covered  
by the 9 sub-meters) adding to the aggregate but absent from individual channels.


In [ ]:
# --- Load quality audit CSV ---
qa_csv = PROJECT_ROOT / "results" / "tables" / "house1_quality_audit.csv"
if qa_csv.exists():
    qa_df = pd.read_csv(qa_csv)
    print("=== Quality Audit Table ===")
    display(qa_df)
else:
    print(f"Quality audit CSV not found at {qa_csv}")

# --- Load cleaning summary CSV ---
cs_csv = PROJECT_ROOT / "results" / "tables" / "house1_cleaning_summary.csv"
if cs_csv.exists():
    cs_df = pd.read_csv(cs_csv)
    print("\n=== Cleaning Summary Table ===")
    display(cs_df)

# --- Sum-of-appliances from audit JSON ---
sac = audit["resampled_1min"]["sum_appliances_comparison"]
print("\n=== Sum-of-Appliances Check ===")
for k, v in sac.items():
    if isinstance(v, float):
        print(f"  {k:45s}: {v:.3f}")
    else:
        print(f"  {k:45s}: {v:,}")

# --- Live validate_cleaned call on small sample ---
print("\n--- Running validate_cleaned on 50 000-row sample ---")
clean_val = pd.read_csv(CLEAN_CSV, nrows=50_000, parse_dates=["Time"])
power_cols = ["Aggregate"] + [f"Appliance{i}" for i in range(1, 10)]
result = cleaning.validate_cleaned(clean_val, power_cols=power_cols)
print("validate_cleaned result (sample):")
for k, v in result.items():
    print(f"  {k}: {v}")


---
## Section 12 — Plots & Export

All key figures are stored in `outputs/figures/` and `results/figures/`.  
Below they are displayed inline.


In [ ]:
fig_paths = {
    "Before/after (1-min aggregate)":
        PROJECT_ROOT / "results" / "figures" / "house1_aggregate_before_after.png",
    "Appliances before/after":
        PROJECT_ROOT / "results" / "figures" / "house1_appliances_before_after.png",
    "Sampling intervals & missingness":
        PROJECT_ROOT / "results" / "figures" / "house1_missingness_diagnostics.png",
    "Duplicate timestamp window":
        PROJECT_ROOT / "outputs" / "figures" / "house1_duplicate_timestamp_window.png",
}

for title, fpath in fig_paths.items():
    print(f"\n--- {title} ---")
    if fpath.exists():
        print(f"  File: {fpath}  ({fpath.stat().st_size:,} bytes)")
        display(Image(filename=str(fpath)))
    else:
        print(f"  ✗ Not found: {fpath}")

# Confirm all result files
print("\n=== Result file inventory ===")
result_files = [
    PROJECT_ROOT / "outputs" / "cleaned" / "CLEAN_House1_1min.csv",
    PROJECT_ROOT / "outputs" / "metrics" / "house1_audit.json",
    PROJECT_ROOT / "results" / "tables" / "house1_cleaning_summary.csv",
    PROJECT_ROOT / "results" / "tables" / "house1_quality_audit.csv",
] + list(fig_paths.values())

for fp in result_files:
    if fp.exists():
        size = fp.stat().st_size
        print(f"  ✓  {fp.relative_to(PROJECT_ROOT)}  ({size:,} bytes)")
    else:
        print(f"  ✗  {fp.relative_to(PROJECT_ROOT)}  — MISSING")


---
## Section 13 — Summary of Findings

### Key numbers

| Metric | Value |
|--------|-------|
| Raw Part 1 rows | 4,377,877 |
| Raw Part 2 rows | 4,140,886 |
| Part 1 exact duplicate rows | 595,935 (13.6 %) |
| Part 2 exact duplicate rows | 0 |
| Part 1 conflicting timestamp groups | 118,191 |
| Part 2 conflicting timestamp groups | 14,362 |
| Overlap window shared timestamps | 191,889 |
| Aggregate conflicts in overlap | 144,267 (75.2 %) |
| Cleaned 1-min bins total | 920,031 |
| Coverage | 2013-10-09 → 2015-07-10 |
| Missing aggregate bins | 119,680 (13.0 %) |
| Longest outage | 59,906 min (Feb 2014) |
| Appliance9 missing | 338,427 (36.8 %) |
| Median sum-agg offset | −159.4 W |
| Mean |sum − agg| | 354.3 W |

### Key findings

- **Part 2 supersedes Part 1** in its coverage window (2014-06-27 onwards); the cleaning strategy  
  correctly uses Part 2 as authoritative and takes only the pre-overlap portion of Part 1.
- **BST artefact:** 48.7 % of Part 1 rows and 48.8 % of Part 2 rows have a Time string that  
  disagrees with the Unix epoch by exactly 1 second — a known British Summer Time logging bug.  
  Using Unix as the authoritative timestamp removes this ambiguity.
- **February 2014 outage:** A single 59,906-minute (~41.6-day) gap accounts for the majority  
  of aggregate missingness.  All 10 channels are equally affected, confirming a system-level  
  (meter or connectivity) failure rather than appliance-level gaps.
- **No imputation:** NaN minute bins are preserved to avoid fabricating power readings.  
  Downstream NILM models must handle missing data explicitly (masking, interpolation, etc.).
- **Sum-of-appliances offset:** The aggregate consistently exceeds the sum of the 9 sub-meters  
  by ~159 W (median), indicating unmeasured loads (lighting circuits, sockets, etc.).

### Remaining limitations

1. **Appliance9 (Electric Heater)** is missing 36.8 % of its minute bins — far more than any  
   other channel.  The root cause is unclear; it may reflect late sensor installation or a  
   different recording cadence for that channel.
2. **No negative or implausible-value removal:** Readings >10 kW are flagged but retained.  
   Users should apply domain-specific screening before training models.
3. **Mean resampling:** 1-minute bins are computed as the mean of all valid sub-minute readings.  
   For appliances with on/off switching within a minute this is an approximation of the  
   true energy consumption.
4. **UK only / single home:** Results are specific to House 1 in a UK detached/semi-detached  
   home context; generalisation to other homes or countries requires re-running the pipeline.

---
*Notebook generated for the Flock Energy NILM coding assignment.*  
*Audit data generated: 2026-11-04T09:24:01 UTC.*
